# Linked List

The program builds a linked list of 6 nodes, each storing a Fibonacci input (38 through 43), then uses parallel programming to compute the Fibonacci value for every node in parallel. A single thread walks the list and spawns one task per node, each task calling the slow recursive `fib()` on its own copy of the pointer `firstprivate(p)`. Once all tasks finish, the results are printed and memory is freed.

### What is a linked list?

A linked list is a chain of nodes where each node holds some data and a pointer to the next node. In this code, each node stores a Fibonacci input (data), its result (fibdata), and a next pointer to the following node. The last node's next is NULL, signaling the end of the chain. Instead of storing elements in contiguous memory like an array, each node lives at a random location in the heap (via malloc) and they are connected only through those next pointers - so to reach node 4 you must walk through nodes 1, 2, and 3 in order.

In [1]:
#include <stdlib.h>
#include <stdio.h>
#include <omp.h>

In [2]:
#ifndef N
#define N 5
#endif
#ifndef FS
#define FS 38
#endif

In [3]:
struct node {
  int data;
  int fibdata;
  struct node *next;
};

Here we can see the 3 main functions used for this program. 

In [4]:
struct node *init_list(struct node *p);
void processwork(struct node *p);
int fib(int n);

### Fibonachi function

`int fib(int n)`: recursively computes the N-th Fibonacci number by breaking it down into fib(n-1) + fib(n-2) until it hits the base cases 0 or 1, then returns the final integer result.

In [5]:
int fib(int n) {
  int x, y;
  if (n < 2) {
    return (n);
  } else {
    x = fib(n - 1);
    y = fib(n - 2);
    return (x + y);
  }
}

### Prosses work

`void processwork(struct node *p)`: takes a single node, computes the Fibonacci number of its data field by calling fib(), and stores the result in its fibdata field. Returns nothing.

In [ ]:
void processwork(struct node *p) {
  int n, temp;
  n = p->data;
  temp = fib(n);

  p->fibdata = temp;
}

### initialize list

`struct node *init_list(struct node *p)`: builds the linked list in memory using malloc, creates N+1 nodes with Fibonacci inputs starting from FS, links them together, and returns a pointer to the head (first node).

In [7]:
struct node *init_list(struct node *p) {
  int i;
  struct node *head = NULL;
  struct node *temp = NULL;

  head = (struct node*) malloc(sizeof(struct node));
  p = head;
  p->data = FS;
  p->fibdata = 0;
  for (i = 0; i < N; i++) {
    temp = (struct node*) malloc(sizeof(struct node));
    p->next = temp;
    p = temp;
    p->data = FS + i + 1;
    p->fibdata = i + 1;
  }

  p->next = NULL;
  return head;
}

## The main code

`main` prints some info messages, sets OpenMP to use the maximum available threads, builds the linked list via `init_list`, then starts a timer. Inside the parallel region, one thread walks the list and creates an independent OpenMP task for each node, while all 8 threads grab and execute those tasks concurrently — each computing its Fibonacci number via `processwork`. Once all tasks finish, the timer stops, the results are printed node by node, memory is freed, and the total compute time is displayed.

In [8]:
int main() {
  double start, end;
  struct node *p = NULL;
  struct node *temp = NULL;
  struct node *head = NULL;

  printf("Process linked list\n");
  printf("  Each linked list node will be processed by function 'processwork()'\n");
  printf("  Each ll node will compute %d fibonacci numbers beginning with %d\n", N, FS);

  omp_set_num_threads(omp_get_max_threads());

  p = init_list(p);
  head = p;

  start = omp_get_wtime();

#pragma omp parallel
  {
#pragma omp master
    printf("Threads:      %d\n", omp_get_num_threads());

#pragma omp single
    {
      p = head;
      while (p) {
#pragma omp task firstprivate(p) // first private is required
        {
          processwork(p);
        }
        p = p->next;
      }
    }
  }

  end = omp_get_wtime();
  p = head;
  while (p != NULL) {
    printf("%d : %d\n", p->data, p->fibdata);
    temp = p->next;
    free(p);
    p = temp;
  }

  free(p);
  printf("Compute Time: %f seconds\n", end - start);

  return 0;
}

In [9]:
main();

Process linked list
  Each linked list node will be processed by function 'processwork()'
  Each ll node will compute 5 fibonacci numbers beginning with 38
Threads:      8
38 : 39088169
39 : 63245986
40 : 102334155
41 : 165580141
42 : 267914296
43 : 433494437
Compute Time: 4.668121 seconds
